# Last.fm Genre Tag Clustering

This notebook transforms normalized genre tags into TF-IDF vectors, reduces their dimensionality using TruncatedSVD and applies K-Means clustering to identify groups of musically similar artist–track combinations.

## 1. Load libraries

In [ ]:
# select env music-clean as it handles TfidfVectorizer and KMeans well, doesn't handle parquet so csv files are used instead

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

## 2. Load normalized dataset

Load the normalized genre-tag dataset produced in the previous notebook.

In [ ]:
model_df = pd.read_csv("../data/processed/lastfm_scrobbles_clean_tags_final_tfid.csv")

## 3. Initial inspection

Verify the dataset before vectorization.

In [ ]:
model_df.head()

In [ ]:
model_df.shape

## 4. Prepare TF-IDF input

Convert lists of normalized genre tags into text documents suitable for TF-IDF vectorization.

In [ ]:
import ast

model_df["tags_filtered"] = model_df["tags_filtered"].apply(ast.literal_eval)

In [ ]:
#TFIDFVectorizer breaks tags with spaces, so we need to join them with underscores

def join_tags(tags):
    return " ".join([t.replace(" ", "_") for t in tags])

model_df["tags_str"] = model_df["tags_filtered"].apply(join_tags)

## Why TF-IDF?

TF-IDF emphasizes distinctive genre tags while downweighting very common labels, making it suitable for identifying meaningful stylistic differences between artist–track combinations.

## 5. TF-IDF vectorization

Transform normalized genre tags into a sparse TF-IDF feature matrix.

In [ ]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(model_df["tags_str"])

In [ ]:
model_df["tags_str"].head(10)

In [ ]:
model_df["tags_str"][5]

## 6. Dimensionality Reduction

Reduce the high-dimensional TF-IDF representation using TruncatedSVD before clustering.

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=50, random_state=42)
X_reduced = svd.fit_transform(X)

## 7. K-Means clustering

Cluster artist–track combinations based on their TF-IDF genre representation.

In [ ]:
kmeans = KMeans(n_clusters=8, random_state=42, n_init=20)
clusters = kmeans.fit_predict(X_reduced)
model_df["cluster"] = clusters

## 8. Initial cluster interpretation

Inspect the most representative tags and artists for each cluster and assign descriptive labels.

In [ ]:
feature_names = vectorizer.get_feature_names_out()

def get_top_tags_per_cluster(X, clusters, feature_names, top_n=10):
    cluster_tags = {}
    
    for cluster in np.unique(clusters):
        idx = clusters == cluster
        mean_vals = np.asarray(X[idx].mean(axis=0)).ravel()
        top_idx = np.argsort(mean_vals)[-top_n:]
        cluster_tags[cluster] = [feature_names[i] for i in top_idx]
        
    return cluster_tags

In [ ]:
top_tags = get_top_tags_per_cluster(X, clusters, feature_names)

for k, v in top_tags.items():
    print(f"Cluster {k}: {v}")

In [ ]:
cluster_names = {
    0: "folk / acoustic",
    1: "classic rock / blues / psychedelic",
    2: "shoegaze / dream pop / indie experimental",
    3: "art pop / neo soul / jazz",
    4: "punk / garage rock",
    5: "britpop / indie rock",
    6: "ambient / chill electronic",
    7: "goth / dark wave / post-punk"
}

In [ ]:
model_df["cluster_name"] = model_df["cluster"].map(cluster_names).fillna("unknown")
model_df["cluster_name"].value_counts()

In [ ]:
model_df.head()

## 8. Validation

Review representative artists within each cluster to assess semantic coherence.

In [ ]:
model_df.groupby("cluster_name")["artist_clean"].value_counts().groupby(level=0).head(3)

## 9. Export clustered dataset

Save the clustered dataset for further validation and visualization.

In [ ]:
model_df.to_csv("../data/processed/lastfm_music_clusters_final.csv", index=False)
model_df.to_parquet("../data/processed/lastfm_music_clusters_final.parquet", index=False)


### Output

`data/processed/lastfm_music_clusters_final.csv`

Used in **06_lastfm_clusters_analysis.ipynb**.

## Notes

This project focuses exclusively on genre-tag representations derived from Last.fm metadata. Audio features were intentionally excluded because the objective was to analyse semantic genre evolution rather than acoustic similarity.